# Calidad del Agua en Embalses de Andalucía — NDCI enmascarado con agua

Flujo orientado a exportación y análisis posterior en QGIS:

1. Generar composites mensuales de **MNDWI** → máscara dinámica de agua  
2. Generar composites mensuales de **NDCI** → proxy de clorofila-a (exclusivo S2)  
3. Aplicar la máscara de agua al NDCI (solo píxeles con MNDWI > 0)  
4. Visualización rápida en geemap  
5. Exportar a Google Drive o GEE Assets con los métodos propios de `NdviSeasonality`  

Las estadísticas por embalse se calculan después en QGIS con **Zonal Statistics**.  
Las celdas sin agua quedan como `nodata` y se ignoran automáticamente.

| Índice | Bandas S2 | Uso |
|--------|-----------|-----|
| MNDWI | Green (B3) / SWIR1 (B11) | Detección dinámica de agua |
| NDCI | Red Edge 1 (B5) / Red (B4) | Proxy de clorofila-a (exclusivo S2) |

## 1. Configuración e importaciones

In [6]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import ee
import geemap
from ndvi2gif import NdviSeasonality

# ee.Authenticate()  # Solo la primera vez en cada máquina
ee.Initialize(project='ee-digdgeografo')
print('Earth Engine inicializado')

Earth Engine inicializado


In [7]:
START_YEAR      = 2020
END_YEAR        = 2025
PERIODS         = 12
SATELLITE       = 'S2'
KEY             = 'median'
WATER_THRESHOLD = 0.0
SCALE           = 20
DRIVE_FOLDER    = 'ndvi2gif_water_quality'

YEARS  = list(range(START_YEAR, END_YEAR + 1))
MONTHS = ['january','february','march','april','may','june',
          'july','august','september','october','november','december']

print(f'Periodo: {START_YEAR}–{END_YEAR} | {SATELLITE} | {SCALE}m')

Periodo: 2020–2025 | S2 | 20m


## 2. Dibuja el área de interés

Usa la herramienta de rectángulo del mapa para dibujar la zona a analizar.  
Luego ejecuta la celda siguiente para capturar la geometría como ROI.

In [8]:
# Mapa centrado en Andalucía — dibuja un rectángulo sobre la zona de interés
draw_map = geemap.Map(center=[37.5, -4.5], zoom=7)
draw_map

Map(center=[37.5, -4.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [9]:
# Captura el rectángulo dibujado como ROI
# Si no has dibujado nada, usa el bounding box de Andalucía por defecto
if draw_map.draw_last_feature:
    roi = draw_map.draw_last_feature.geometry()
    print('ROI capturada del mapa')
else:
    roi = ee.Geometry.BBox(-7.5, 36.0, -1.6, 38.8)
    print('Sin dibujo detectado — usando bounding box de Andalucía por defecto')

print(f'ROI bounds: {roi.bounds().getInfo()}')

ROI capturada del mapa
ROI bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-7.481689, 36.035913], [-1.922607, 36.035913], [-1.922607, 38.664201], [-7.481689, 38.664201], [-7.481689, 36.035913]]]}


## 3. Composites mensuales MNDWI y NDCI

In [11]:
s2_mndwi = NdviSeasonality(
    roi=roi,
    periods=PERIODS,
    start_year=START_YEAR,
    end_year=END_YEAR,
    sat=SATELLITE,
    key=KEY,
    index='mndwi'
)
composite_mndwi = s2_mndwi.get_year_composite()
print(f'MNDWI: {composite_mndwi.size().getInfo()} imágenes | bandas: {composite_mndwi.first().bandNames().getInfo()}')

There we go again...
Applying cloud filter to Sentinel-2: max 20% cloud cover
Using MODIS Terra + Aqua LST (maximum coverage)
Using all Sentinel-1 orbits (ascending + descending).
Applying S1 ARD preprocessing:
  - Speckle filter: REFINED_LEE
  - Terrain correction: True
  - Terrain model: VOLUME
Sentinel-2 collection configured with cloud filtering
Year 2020: Successfully processed 12 periods using mndwi index
Year 2021: Successfully processed 12 periods using mndwi index
Year 2022: Successfully processed 12 periods using mndwi index
Year 2023: Successfully processed 12 periods using mndwi index
Year 2024: Successfully processed 12 periods using mndwi index
Year 2025: Successfully processed 12 periods using mndwi index
MNDWI: 6 imágenes | bandas: ['january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november', 'december']


In [12]:
s2_ndci = NdviSeasonality(
    roi=roi,
    periods=PERIODS,
    start_year=START_YEAR,
    end_year=END_YEAR,
    sat=SATELLITE,
    key=KEY,
    index='ndci'
)
composite_ndci = s2_ndci.get_year_composite()
print(f'NDCI:  {composite_ndci.size().getInfo()} imágenes | bandas: {composite_ndci.first().bandNames().getInfo()}')

There we go again...
Applying cloud filter to Sentinel-2: max 20% cloud cover
Using MODIS Terra + Aqua LST (maximum coverage)
Using all Sentinel-1 orbits (ascending + descending).
Applying S1 ARD preprocessing:
  - Speckle filter: REFINED_LEE
  - Terrain correction: True
  - Terrain model: VOLUME
Sentinel-2 collection configured with cloud filtering
Year 2020: Successfully processed 12 periods using ndci index
Year 2021: Successfully processed 12 periods using ndci index
Year 2022: Successfully processed 12 periods using ndci index
Year 2023: Successfully processed 12 periods using ndci index
Year 2024: Successfully processed 12 periods using ndci index
Year 2025: Successfully processed 12 periods using ndci index
NDCI:  6 imágenes | bandas: ['january', 'february', 'march', 'april', 'may', 'june', 'july', 'august', 'september', 'october', 'november', 'december']


In [13]:
# Listas para acceder por índice de año
mndwi_list = composite_mndwi.toList(composite_mndwi.size())
ndci_list  = composite_ndci.toList(composite_ndci.size())

# NDCI enmascarado: solo píxeles con MNDWI > umbral
ndci_water = ee.ImageCollection([
    ee.Image(ndci_list.get(i))
      .updateMask(ee.Image(mndwi_list.get(i)).gt(WATER_THRESHOLD))
      .set('year', year)
    for i, year in enumerate(YEARS)
])
print(f'NDCI enmascarado: {ndci_water.size().getInfo()} imágenes')

NDCI enmascarado: 6 imágenes


## 4. Visualización en geemap

In [15]:
YEAR_VIZ  = 2023
MONTH_VIZ = 'january'

idx = YEARS.index(YEAR_VIZ)
img_mndwi      = ee.Image(mndwi_list.get(idx)).select(MONTH_VIZ)
img_ndci       = ee.Image(ndci_list.get(idx)).select(MONTH_VIZ)
img_ndci_water = ee.Image(ndci_water.toList(ndci_water.size()).get(idx)).select(MONTH_VIZ)

ndci_palette = ['#053061','#4393c3','#f7f7f7','#f4a582','#b2182b']

Map = geemap.Map()
Map.centerObject(roi, zoom=8)
Map.addLayer(
    img_mndwi,
    {'min': -0.3, 'max': 0.6, 'palette': ['#d73027','#fee090','#e0f3f8','#313695']},
    f'MNDWI {MONTH_VIZ} {YEAR_VIZ}'
)
Map.addLayer(
    img_mndwi.gt(WATER_THRESHOLD),
    {'min': 0, 'max': 1, 'palette': ['white', '#2166ac']},
    f'Máscara agua {MONTH_VIZ} {YEAR_VIZ}', False
)
Map.addLayer(
    img_ndci,
    {'min': -0.2, 'max': 0.3, 'palette': ndci_palette},
    f'NDCI sin máscara {MONTH_VIZ} {YEAR_VIZ}', False
)
Map.addLayer(
    img_ndci_water,
    {'min': -0.2, 'max': 0.3, 'palette': ndci_palette},
    f'NDCI solo agua {MONTH_VIZ} {YEAR_VIZ}'
)
Map

Map(center=[37.35322787362771, -4.7021480000000295], controls=(WidgetControl(options=['position', 'transparent…

## 5a. Exportar a Google Drive

In [16]:
tasks = []                                                                                        
ndci_water_list = ndci_water.toList(ndci_water.size())                                            
                                                                                                
for i, year in enumerate(YEARS):                       
  img = ee.Image(ndci_water_list.get(i))                                                        
  task = s2_ndci.export_to_drive(                                                               
      image=img,
      description=f'NDCI_agua_{year}',
      folder=DRIVE_FOLDER,
      scale=SCALE,
      crs='EPSG:32630'
  )
  tasks.append(task)
  print(f'{year} → tarea lanzada: {task.id}')

print(f'\n{len(tasks)} tareas enviadas → carpeta Drive: "{DRIVE_FOLDER}"')
print('Progreso: https://code.earthengine.google.com/tasks')

2020 → tarea lanzada: KFJIO7UP24DUVNDM3PRN5GGU
2021 → tarea lanzada: C73BCEVLKXQ4CNX7X7EEUDKN
2022 → tarea lanzada: XKENKJZRHRT26JLPUW5WAOEW
2023 → tarea lanzada: H36GVMYJICN45BMA4K7WRKUO
2024 → tarea lanzada: DSN43CLIH6KNR7GNTWYNSTRM
2025 → tarea lanzada: MXMKMSAIXRSEVFSSX4B2GLC2

6 tareas enviadas → carpeta Drive: "ndvi2gif_water_quality"
Progreso: https://code.earthengine.google.com/tasks


## 5b. Alternativa: exportar como GEE Asset

In [ ]:
# GEE_ASSET_FOLDER = 'projects/ee-digdgeografo/assets/water_quality'

# for i, year in enumerate(YEARS):
#     img = ee.Image(ndci_water_list.get(i))
#     task = s2_ndci.export_to_asset(
#         image=img,
#         description=f'NDCI_agua_asset_{year}',
#         asset_id=f'{GEE_ASSET_FOLDER}/ndci_agua_embalses_{year}',
#         scale=SCALE,
#         crs='EPSG:4326'
#     )
#     print(f'{year} → asset lanzado: {task.id}')

## 6. Workflow en QGIS

Una vez descargados los GeoTIFFs de Google Drive:

1. Cargar el shapefile de embalses y el GeoTIFF del año a analizar
2. **Raster → Zonal Statistics** (o caja de herramientas de Processing)
   - Raster layer: `NDCI_agua_2023.tif`
   - Vector layer: shapefile de embalses
   - Statistics: Mean, Median, StdDev, Count
3. Los embalses sin agua en ese periodo tendrán `NULL` → descartarlos en el análisis

| NDCI | Interpretación |
|------|----------------|
| < 0  | Agua limpia / baja clorofila |
| 0 – 0.1 | Concentración moderada |
| 0.1 – 0.2 | Elevada (vigilar) |
| > 0.2 | Posible floración algal (alerta) |